# Stage 3 — Vibration Feature Extraction

This notebook extracts time-domain and frequency-domain features
from the Paderborn vibration signals.

Each measurement file corresponds to one bearing, one operating
condition, and one measurement.

In [ ]:
# ============================================================
# FEATURE EXTRACTION SETUP
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io as sio

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Use one known measurement file for the initial test
sample_file = next(RAW_DIR.glob("K001/*.mat"))

print("Sample file:")
print(sample_file)

In [ ]:
# ============================================================
# INSPECT ONE MATLAB FILE
# ============================================================

mat = sio.loadmat(
    sample_file,
    squeeze_me=False,
    struct_as_record=False
)

print("Top-level variables:")
print(
    [
        key for key in mat.keys()
        if not key.startswith("__")
    ]
)

In [ ]:
# ============================================================
# INSPECT MATLAB STRUCTURE AFTER UNWRAPPING
# ============================================================

data_key = [
    key for key in mat.keys()
    if not key.startswith("__")
][0]

# Unwrap the (1, 1) NumPy array
data = mat[data_key][0, 0]

print("Data key:", data_key)
print("Data type:", type(data))

print("\nAvailable fields:")
for field in data._fieldnames:
    print("-", field)

In [ ]:
# ============================================================
# INSPECT X AND Y STRUCTURES
# ============================================================

X = data.X
Y = data.Y

print("X type:", type(X))
print("Y type:", type(Y))

print("\nX fields:")
if hasattr(X, "_fieldnames"):
    for field in X._fieldnames:
        print("-", field)

print("\nY fields:")
if hasattr(Y, "_fieldnames"):
    for field in Y._fieldnames:
        print("-", field)

In [ ]:
# ============================================================
# INSPECT ALL Y CHANNELS
# ============================================================

Y_channels = np.asarray(data.Y).flatten()

print("Number of Y channels:", len(Y_channels))
print("Y array shape:", np.asarray(data.Y).shape)

print("\nAvailable Y channels:")

for i, channel in enumerate(Y_channels):

    name = channel.Name

    # Convert MATLAB string array to Python string
    if isinstance(name, np.ndarray):
        name = str(name.flatten()[0])

    data_shape = np.asarray(channel.Data).shape

    print(
        f"{i}: {name} | Data shape: {data_shape}"
    )

In [ ]:
# ============================================================
# EXTRACT VIBRATION_1 ROBUSTLY
# ============================================================

def get_vibration_signal(mat_file):
    """
    Load one Paderborn .mat file and return the vibration_1 signal.
    """

    mat = sio.loadmat(
        mat_file,
        squeeze_me=False,
        struct_as_record=False
    )

    # Find the actual measurement variable
    data_keys = [
        key for key in mat.keys()
        if not key.startswith("__")
    ]

    if len(data_keys) != 1:
        raise ValueError(
            f"Unexpected MATLAB structure in {mat_file}"
        )

    data = mat[data_keys[0]][0, 0]

    # Find vibration channel by name
    y_channels = np.asarray(data.Y).flatten()

    for channel in y_channels:

        name = channel.Name

        if isinstance(name, np.ndarray):
            name = str(name.flatten()[0])

        if name == "vibration_1":

            signal = np.asarray(
                channel.Data
            ).squeeze()

            return signal.astype(np.float64)

    raise ValueError(
        f"vibration_1 channel not found in {mat_file}"
    )

In [ ]:
# ============================================================
# TEST VIBRATION EXTRACTION
# ============================================================

test_signal = get_vibration_signal(sample_file)

print("Signal extracted successfully")
print("Number of samples:", len(test_signal))
print("Minimum:", test_signal.min())
print("Maximum:", test_signal.max())
print("Mean:", test_signal.mean())
print("Standard deviation:", test_signal.std())

In [ ]:
# ============================================================
# FEATURE EXTRACTION FUNCTION
# ============================================================

from scipy.stats import kurtosis, skew
import numpy as np


FS = 64000  # Paderborn vibration sampling frequency


def extract_vibration_features(signal, fs=FS):
    """
    Extract time-domain and frequency-domain features
    from one vibration signal.
    """

    signal = np.asarray(signal, dtype=np.float64).flatten()

    if len(signal) == 0:
        raise ValueError("Signal is empty.")

    # --------------------------------------------------------
    # Time-domain features
    # --------------------------------------------------------

    mean_value = np.mean(signal)
    std_value = np.std(signal)

    rms_value = np.sqrt(
        np.mean(signal ** 2)
    )

    min_value = np.min(signal)
    max_value = np.max(signal)

    peak_to_peak = max_value - min_value

    kurtosis_value = kurtosis(
        signal,
        fisher=True,
        bias=False
    )

    skewness_value = skew(
        signal,
        bias=False
    )

    crest_factor = (
        np.max(np.abs(signal)) / rms_value
        if rms_value != 0 else 0
    )

    # --------------------------------------------------------
    # Frequency-domain features
    # --------------------------------------------------------

    n = len(signal)

    # Remove DC component before spectral analysis
    centered_signal = signal - mean_value

    fft_values = np.fft.rfft(centered_signal)
    frequencies = np.fft.rfftfreq(
        n,
        d=1 / fs
    )

    magnitude = np.abs(fft_values)

    # Ignore DC bin when finding dominant frequency
    magnitude_no_dc = magnitude.copy()

    if len(magnitude_no_dc) > 1:
        magnitude_no_dc[0] = 0

    dominant_index = np.argmax(
        magnitude_no_dc
    )

    dominant_frequency = frequencies[
        dominant_index
    ]

    dominant_magnitude = magnitude[
        dominant_index
    ]

    # Power spectrum
    power = magnitude ** 2

    total_power = np.sum(power)

    if total_power > 0:

        spectral_centroid = (
            np.sum(frequencies * power)
            / total_power
        )

        spectral_bandwidth = np.sqrt(
            np.sum(
                ((frequencies - spectral_centroid) ** 2)
                * power
            )
            / total_power
        )

        probability = power / total_power

        spectral_entropy = -np.sum(
            probability[
                probability > 0
            ]
            * np.log2(
                probability[
                    probability > 0
                ]
            )
        )

    else:

        spectral_centroid = 0.0
        spectral_bandwidth = 0.0
        spectral_entropy = 0.0

    spectral_rms = np.sqrt(
        np.mean(power)
    )

    # --------------------------------------------------------
    # Frequency band powers
    # --------------------------------------------------------

    bands = [
        (0, 500),
        (500, 1000),
        (1000, 2000),
        (2000, 3000),
        (3000, 4000),
        (4000, 5000),
        (5000, 6000),
        (6000, 7000),
        (7000, 8000),
        (8000, 9000),
        (9000, 10000)
    ]

    features = {
        "mean": mean_value,
        "std": std_value,
        "rms": rms_value,
        "min": min_value,
        "max": max_value,
        "peak_to_peak": peak_to_peak,
        "kurtosis": kurtosis_value,
        "skewness": skewness_value,
        "crest_factor": crest_factor,
        "dominant_frequency": dominant_frequency,
        "dominant_magnitude": dominant_magnitude,
        "spectral_centroid": spectral_centroid,
        "spectral_bandwidth": spectral_bandwidth,
        "spectral_rms": spectral_rms,
        "spectral_entropy": spectral_entropy
    }

    for low, high in bands:

        mask = (
            (frequencies >= low)
            & (frequencies < high)
        )

        features[
            f"band_power_{low}_{high}"
        ] = np.sum(
            power[mask]
        )

    return features

In [ ]:
# ============================================================
# TEST FEATURE EXTRACTION ON ONE SIGNAL
# ============================================================

sample_features = extract_vibration_features(
    test_signal
)

feature_test_df = pd.DataFrame(
    [sample_features]
)

print(
    "Number of extracted features:",
    len(sample_features)
)

display(feature_test_df.T)

In [ ]:
# ============================================================
# TEST REPRESENTATIVE FILE FROM EACH CATEGORY
# ============================================================

representative_bearings = {
    "healthy": "K001",
    "outer": "KA01",
    "inner": "KI01",
    "both": "KB23"
}

representative_results = []

for category, bearing in representative_bearings.items():

    file = next(
        (RAW_DIR / bearing).glob("*.mat")
    )

    signal = get_vibration_signal(file)

    features = extract_vibration_features(
        signal
    )

    features["bearing"] = bearing
    features["category"] = category
    features["file"] = file.name

    representative_results.append(
        features
    )

representative_df = pd.DataFrame(
    representative_results
)

print(
    "Representative files processed:",
    len(representative_df)
)

display(
    representative_df[
        ["bearing", "category", "file"]
    ]
)

In [ ]:
# ============================================================
# BATCH FEATURE EXTRACTION - DIAGNOSTIC VERSION
# ============================================================

from pathlib import Path
import time

# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

metadata_path = PROCESSED_DIR / "bearing_metadata.csv"
bearing_metadata = pd.read_csv(metadata_path)

# ------------------------------------------------------------
# Collect all MATLAB files
# ------------------------------------------------------------

mat_files = sorted(
    RAW_DIR.rglob("*.mat")
)

print("Total .mat files found:", len(mat_files))

if len(mat_files) != 2560:
    raise ValueError(
        f"Expected 2560 .mat files, found {len(mat_files)}"
    )

# ------------------------------------------------------------
# Extract features
# ------------------------------------------------------------

all_features = []
failed_files = []

start_time = time.time()

for i, mat_file in enumerate(mat_files, start=1):

    try:

        # Extract vibration signal
        signal = get_vibration_signal(mat_file)

        # Extract features
        features = extract_vibration_features(signal)

        # ----------------------------------------------------
        # Parse filename
        # ----------------------------------------------------

        parts = mat_file.stem.split("_")

        if len(parts) < 5:
            raise ValueError(
                f"Unexpected filename format: {mat_file.name}"
            )

        operating_condition = "_".join(parts[:3])
        bearing = parts[3]
        measurement_number = int(parts[4])

        # ----------------------------------------------------
        # Get metadata
        # ----------------------------------------------------

        metadata_row = bearing_metadata[
            bearing_metadata["bearing"] == bearing
        ]

        if len(metadata_row) != 1:
            raise ValueError(
                f"Metadata not found uniquely for {bearing}"
            )

        metadata_row = metadata_row.iloc[0]

        # ----------------------------------------------------
        # Add metadata to features
        # ----------------------------------------------------

        features["file"] = mat_file.name
        features["bearing"] = bearing
        features["operating_condition"] = operating_condition
        features["measurement_number"] = measurement_number

        features["fault_location"] = (
            metadata_row["fault_location"]
        )

        features["fault_class"] = int(
            metadata_row["fault_class"]
        )

        features["binary_label"] = int(
            metadata_row["binary_label"]
        )

        all_features.append(features)

    except Exception as e:

        failed_files.append({
            "file": mat_file.name,
            "path": str(mat_file),
            "error_type": type(e).__name__,
            "error": str(e)
        })

        print(
            f"\nFAILED FILE #{i}: {mat_file.name}"
        )
        print(
            f"Error: {type(e).__name__}: {e}"
        )

    # --------------------------------------------------------
    # Progress report
    # --------------------------------------------------------

    if i % 100 == 0 or i == len(mat_files):

        elapsed = time.time() - start_time

        print(
            f"Processed {i}/{len(mat_files)} files "
            f"({i / len(mat_files) * 100:.1f}%) "
            f"| successful: {len(all_features)} "
            f"| failed: {len(failed_files)} "
            f"| elapsed: {elapsed / 60:.1f} min"
        )

# ------------------------------------------------------------
# Create DataFrames
# ------------------------------------------------------------

feature_df = pd.DataFrame(all_features)
failed_df = pd.DataFrame(failed_files)

print("\n==============================================")
print("EXTRACTION FINISHED")
print("==============================================")

print("Successful files:", len(feature_df))
print("Failed files:", len(failed_df))

if len(failed_df) > 0:
    print("\nFailed files:")
    display(failed_df)


In [ ]:
# ============================================================
# SAVE SUCCESSFUL FEATURES AND FAILED-FILE LOG
# ============================================================

feature_output_path = (
    PROCESSED_DIR / "paderborn_vibration_features.csv"
)

failure_output_path = (
    PROCESSED_DIR / "failed_mat_files.csv"
)

# Save successfully processed files
feature_df.to_csv(
    feature_output_path,
    index=False
)

# Save failed-file provenance
failed_df.to_csv(
    failure_output_path,
    index=False
)

print("Successful feature rows:", len(feature_df))
print("Failed files:", len(failed_df))

print("\nFeature dataset saved:")
print(feature_output_path.resolve())

print("\nFailure log saved:")
print(failure_output_path.resolve())

In [ ]:
# ============================================================
# VERIFY FINAL FEATURE DATASET
# ============================================================

print("Total feature rows:", len(feature_df))
print("Unique bearings:", feature_df["bearing"].nunique())
print(
    "Unique operating conditions:",
    feature_df["operating_condition"].nunique()
)

print("\nRows per bearing:")
print(
    feature_df.groupby("bearing").size()
)

print("\nFault-location distribution:")
print(
    feature_df["fault_location"].value_counts()
)

print("\nMissing values:")
print(
    feature_df.isna().sum().sum()
)

print("\nFailed files:")
display(failed_df)